<a href="https://colab.research.google.com/github/Julia-Susser/AI-Startup-Impact-on-US-Labor-Economy-via-Task-Automation/blob/master/kaggle_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
pip install lightly

In [11]:
import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as T
import torch.nn.functional as F

import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import kagglehub
import os
from natsort import natsorted
from torch.utils.data import Dataset, DataLoader, ConcatDataset
from tqdm import tqdm
from google.colab import files
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import models, transforms
from lightly.loss import NTXentLoss
from lightly.models.modules.heads import SimCLRProjectionHead
from lightly.transforms import SimCLRTransform
from lightly.data import LightlyDataset
from sklearn.preprocessing import StandardScaler

from torchvision.transforms import RandomResizedCrop, RandomHorizontalFlip, RandomVerticalFlip
from torchvision import transforms
import torchvision.transforms as T

from sklearn.model_selection import KFold
import torch.optim as optim
import timm

import matplotlib.pyplot as plt
import random

import timm
from sklearn.metrics import r2_score
from sklearn.preprocessing import StandardScaler
import albumentations as A
from albumentations.pytorch import ToTensorV2

dtype = torch.float32
cuda0= torch.device("cuda:0")


DOWNLOAD DATASETS


In [4]:

# # https://www.kaggle.com/datasets/jonasdahlqvist/grass-nograss-dataset
# grass_dir1 = kagglehub.dataset_download("jonasdahlqvist/grass-nograss-dataset")


# # https://www.kaggle.com/datasets/timofeymoiseev/grass
# grass_dir2 = kagglehub.dataset_download("timofeymoiseev/grass")

# path = kagglehub.dataset_download("usharengaraju/grassclover-dataset")
# print("Path to dataset files:", path)
competition_dir = "csiro-biomass"

In [5]:
files.upload()
!mkdir -p ~/.kaggle
!mv kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

Saving kaggle.json to kaggle.json


In [6]:
!kaggle competitions download -c csiro-biomass
!unzip -q csiro-biomass.zip -d ./csiro-biomass
!ls ./csiro-biomass


 96% 0.98G/1.02G [00:13<00:01, 28.2MB/s]
100% 1.02G/1.02G [00:13<00:00, 81.1MB/s]
sample_submission.csv  test  test.csv  train  train.csv


In [5]:
from google.colab import drive
drive.mount('/content/drive',force_remount=True)
competition_dir = '/content/drive/MyDrive/datasets/biomass_competition'
competition_img_dir = f"{competition_dir}/train/"
df = pd.read_csv(f"{competition_dir}/train.csv")

Mounted at /content/drive


In [6]:
df = data = pd.read_csv(competition_dir+"/train.csv")
target_cols = list(data.target_name.unique())
df["base_id"] = df["sample_id"].str.split("__").str[0] ## extract the id

# Multiple targets for each sample
df = (
    df.pivot_table(
        index=[
            "base_id",
            "image_path",
            "Sampling_Date",
            "State",
            "Species",
            "Pre_GSHH_NDVI",
            "Height_Ave_cm",
        ],
        columns="target_name",
        values="target",
    )
    .reset_index()
)
df.columns.name = None


print(df.head())
print("\nColumns:")
print(df.columns.tolist())


        base_id              image_path Sampling_Date State  \
0  ID1011485656  train/ID1011485656.jpg      2015/9/4   Tas   
1  ID1012260530  train/ID1012260530.jpg      2015/4/1   NSW   
2  ID1025234388  train/ID1025234388.jpg      2015/9/1    WA   
3  ID1028611175  train/ID1028611175.jpg     2015/5/18   Tas   
4  ID1035947949  train/ID1035947949.jpg     2015/9/11   Tas   

             Species  Pre_GSHH_NDVI  Height_Ave_cm  Dry_Clover_g  Dry_Dead_g  \
0    Ryegrass_Clover           0.62         4.6667        0.0000     31.9984   
1            Lucerne           0.55        16.0000        0.0000      0.0000   
2  SubcloverDalkeith           0.38         1.0000        6.0500      0.0000   
3           Ryegrass           0.66         5.0000        0.0000     30.9703   
4           Ryegrass           0.54         3.5000        0.4343     23.2239   

   Dry_Green_g  Dry_Total_g    GDM_g  
0      16.2751      48.2735  16.2750  
1       7.6000       7.6000   7.6000  
2       0.0000       6.

In [ ]:
MODEL_NAME = 'convnext_tiny'
PRETRAINED = True

IMG_SIZE = 1024
IMG_FOLDER=competition_img_dir
BATCH_SIZE = 4
EPOCHS = 30
FREEZE_EPOCHS = 10
LEARNING_RATE = 1e-4
FINETUNE_LR = 1e-5
NUM_WORKERS= 2


device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

target_cols_train = ['Dry_Total_g', 'GDM_g', 'Dry_Green_g']
target_cols_eval  = ['Dry_Green_g', 'Dry_Dead_g', 'Dry_Clover_g', 'GDM_g', 'Dry_Total_g']

loss_weights = {'total_loss': 0.5, 'gdm_loss': 0.2, 'green_loss': 0.1}
r2_weights    = [0.1, 0.1, 0.1, 0.2, 0.5]


# ---------------------------
# Augmentation
# ---------------------------
class AugmentationFactory:
    def __init__(self):
        self.img_size = IMG_SIZE

    def get_train_transforms(self):
        return A.Compose([
            A.HorizontalFlip(p=0.5),
            A.VerticalFlip(p=0.5),
            A.RandomRotate90(p=0.5),
            A.ColorJitter(0.2, 0.2, 0.2, 0.1, p=0.75),
            A.Resize(self.img_size, self.img_size),
            A.Normalize(mean=[0.485, 0.456, 0.406],
                        std=[0.229, 0.224, 0.225]),
            ToTensorV2()
        ])

    def get_valid_transforms(self):
        return A.Compose([
            A.Resize(self.img_size, self.img_size),
            A.Normalize(mean=[0.485, 0.456, 0.406],
                        std=[0.229, 0.224, 0.225]),
            ToTensorV2()
        ])

# ---------------------------
# Dataset
# ---------------------------
class GrassLabeledDataset(Dataset):
    """
    Dataset that returns image, labels_3 (train targets), labels_5 (eval targets).
    """
    def __init__(self, transform=None):
        self.transform = transform
        self.df = df
        self.img_folder = IMG_FOLDER

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_path = os.path.join(self.img_folder, f"{row['base_id']}.jpg")
        img = np.array(Image.open(img_path).convert('RGB'))
        if self.transform:
            img = self.transform(image=img)['image']
        labels_3 = torch.tensor(
            row[target_cols_train].astype(float).values, dtype=torch.float32
        )
        labels_5 = torch.tensor(row.loc[target_cols_eval].astype(float).values, dtype=torch.float32)
        return img, labels_3, labels_5


augmentor = AugmentationFactory()


train_dataset = GrassLabeledDataset(
    transform=augmentor.get_train_transforms()
)

val_dataset = GrassLabeledDataset(
    transform=augmentor.get_valid_transforms()
)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=num_workers, pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False, num_workers=num_workers, pin_memory=True)



In [40]:


# ---------------------------
# Weighted Loss
# ---------------------------
class WeightedBiomassLoss(nn.Module):
    def __init__(self, w):
        super().__init__()
        self.fn = nn.SmoothL1Loss()
        self.w = w

    def forward(self, preds, y):
        p_total, p_gdm, p_green = preds
        return (
            self.w['total_loss'] * self.fn(p_total, y[:, [0]]) +
            self.w['gdm_loss']   * self.fn(p_gdm,   y[:, [1]]) +
            self.w['green_loss'] * self.fn(p_green, y[:, [2]])
        )

# ---------------------------
# R² Scorer (modular with shared preds_5 function)
# ---------------------------
class CompetitionScorer:
    def __init__(self, weights):
        self.weights = np.array(weights)

    def _build_preds_5(self, preds_dict):
        pred_total = preds_dict['total']
        pred_gdm   = preds_dict['gdm']
        pred_green = preds_dict['green']
        pred_clover = np.maximum(0, pred_gdm - pred_green)
        pred_dead   = np.maximum(0, pred_total - pred_gdm)
        preds_5 = np.stack(
            [pred_green, pred_dead, pred_clover, pred_gdm, pred_total], axis=1
        )
        return preds_5

    def r2(self, preds_dict, targets_5):
        preds_5 = self._build_preds_5(preds_dict)
        r2s = r2_score(targets_5, preds_5, multioutput='raw_values')
        weighted_r2 = float(np.sum(r2s * self.weights))
        return weighted_r2

    def mse(self, preds_dict, targets_5):
        preds_5 = self._build_preds_5(preds_dict)
        mse = float(np.mean((preds_5 - targets_5) ** 2))
        return mse


# ---------------------------
# Evaluate and Train functions
# ---------------------------
def evaluate(model, loader, criterion, scorer):
    model.eval()
    val_loss = 0
    preds = {'total': [], 'gdm': [], 'green': []}
    trues_5 = []

    with torch.no_grad():
        for x, y3, y5 in tqdm(loader, desc="Evaluating validation set", unit="batch"):
            x, y3, y5 = x.to(device), y3.to(device), y5.to(device)
            _, preds_tuple = model(x)
            loss = criterion(preds_tuple, y3)
            val_loss += loss.item()
            preds['total'].append(preds_tuple[0].cpu().numpy())
            preds['gdm'].append(preds_tuple[1].cpu().numpy())
            preds['green'].append(preds_tuple[2].cpu().numpy())
            trues_5.append(y5.cpu().numpy())

    preds = {k: np.concatenate(v).flatten() for k, v in preds.items()}
    trues_5 = np.concatenate(trues_5)

    weighted_r2 = scorer.r2(preds, trues_5)
    mse = scorer.mse(preds, trues_5)
    return val_loss / len(loader), weighted_r2, mse


def train_two_stage(model, train_loader, val_loader, criterion, scorer):
    model.to(device)
    # Stage1: freeze backbone
    for p in model.backbone.parameters():
        p.requires_grad = False
    opt = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=LEARNING_RATE)
    best = -float('inf')

    for epoch in range(1, FREEZE_EPOCHS+1):
        model.train()
        loss_sum = 0
        for x, y3, _ in tqdm(train_loader, desc=f"Stage1 Epoch {epoch}"):
            x, y3 = x.to(device), y3.to(device)
            opt.zero_grad()
            out, preds_tuple = model(x)
            loss = criterion(preds_tuple, y3)
            loss.backward()
            opt.step()
            loss_sum += loss.item()
        val_loss, val_r2, val_mse = evaluate(model, val_loader, criterion, scorer)
        print(f"[Stage1 E{epoch}] Train={loss_sum/len(train_loader):.4f} | Val={val_loss:.4f} | R²={val_r2:.4f}")
        best = max(best, val_r2)

    # Stage2: fine-tune all
    for p in model.backbone.parameters():
        p.requires_grad = True
    opt = optim.Adam(model.parameters(), lr=FINETUNE_LR)

    for epoch in range(FREEZE_EPOCHS+1, EPOCHS+1):
        model.train()
        loss_sum = 0
        for x, y3, _ in tqdm(train_loader, desc=f"Stage2 Epoch {epoch}"):
            x, y3 = x.to(device), y3.to(device)
            opt.zero_grad()
            out, preds_tuple = model(x)
            loss = criterion(preds_tuple, y3)
            loss.backward()
            opt.step()
            loss_sum += loss.item()
        val_loss, val_r2, val_mse = evaluate(model, val_loader, criterion, scorer)
        print(f"[Stage2 E{epoch}] Train={loss_sum/len(train_loader):.4f} | Val={val_loss:.4f} | R²={val_r2:.4f}")
        best = max(best, val_r2)

    print(f"Training complete. Best R²={best:.4f}")
    return model


def train_one_stage(model, train_loader, val_loader, criterion, scorer):
    model.to(device)
    opt = optim.Adam(model.parameters(), lr=LEARNING_RATE)
    best = -float('inf')
    for epoch in range(1, EPOCHS + 1):
        model.train()
        loss_sum = 0

        # ---- Training loop ----
        for x, y3, _ in tqdm(train_loader, desc=f"Epoch {epoch}", unit="batch"):
            x, y3 = x.to(device), y3.to(device)
            opt.zero_grad()
            _, preds_tuple = model(x)
            loss = criterion(preds_tuple, y3)
            loss.backward()
            opt.step()
            loss_sum += loss.item()

        # ---- Validation ----
        val_loss, val_r2, val_mse = evaluate(model, val_loader, criterion, scorer)
        print(f"[Epoch {epoch}] Train={loss_sum/len(train_loader):.4f} | "
              f"Val={val_loss:.4f} | MSE={val_mse:.4f} | R²={val_r2:.4f}")

        best = max(best, val_r2)
    print(f"Training complete. Best R²={best:.4f}")
    return model


def validate(model, val_loader, criterion, scorer, n_show=5):
    # ---- Run evaluation first ----
    val_loss, r2, val_mse = evaluate(model, val_loader, criterion, scorer)
    print(f"\nValidation Results:")
    print(f"Val Loss = {val_loss:.4f}")
    print(f"Val MSE = {val_mse:.4f}")
    print(f"Weighted R² = {r2:.4f}\n")

    # ---- Pick random samples ----
    total_samples = len(val_loader.dataset)
    sample_indices = np.random.choice(total_samples, n_show, replace=False)
    col_names = ["Dry_Green_g", "Dry_Dead_g", "Dry_Clover_g", "GDM_g", "Dry_Total_g"]

    print(f"Showing {n_show} random validation samples...\n")

    model.eval()
    model.to(device)

    # tqdm progress bar for how long visualization takes
    for i in tqdm(range(n_show), desc="Visualizing samples", unit="img"):
        idx = sample_indices[i]
        x, y3, y5 = val_loader.dataset[idx]
        x_in = x.unsqueeze(0).to(device)  # [1, 3, H, W]

        with torch.no_grad():
            _, preds_tuple = model(x_in)

        # Extract predictions
        pred_total = preds_tuple[0].cpu().item()
        pred_gdm   = preds_tuple[1].cpu().item()
        pred_green = preds_tuple[2].cpu().item()
        pred_clover = max(0, pred_gdm - pred_green)
        pred_dead   = max(0, pred_total - pred_gdm)
        preds_5 = [pred_green, pred_dead, pred_clover, pred_gdm, pred_total]

        true_5 = y5.numpy()

        # ---- Print comparison ----
        print(f"\n--- Sample {i+1} ---")
        for j, name in enumerate(col_names):
            print(f"{name:<12}: True={true_5[j]:7.2f} | Pred={preds_5[j]:7.2f}")

        # ---- Show image ----
        img = x.permute(1, 2, 0).numpy()
        img = (img - img.min()) / (img.max() - img.min())
        plt.imshow(img)
        plt.axis("off")
        plt.show()




In [35]:
# ---------------------------
# Model (two-stream split inside model)
# ---------------------------
class TwoStreamConvNeXt3Head(nn.Module):
    def __init__(self, backbone_name="convnext_tiny", pretrained=True):
        super().__init__()
        self.backbone = timm.create_model(
            backbone_name, pretrained=pretrained, num_classes=0, global_pool="avg"
        )
        n_feat = self.backbone.num_features
        n_combined = n_feat * 2

        def make_head():
            return nn.Sequential(
                nn.Linear(n_combined, n_combined // 2),
                nn.ReLU(inplace=True),
                nn.Dropout(0.3),
                nn.Linear(n_combined // 2, 1)
            )

        self.head_total = make_head()
        self.head_gdm   = make_head()
        self.head_green = make_head()

    def forward(self, x):
        B, C, H, W = x.shape
        mid = W // 2
        left  = x[:, :, :, :mid]
        right = x[:, :, :, mid:]
        f_left  = self.backbone(left)
        f_right = self.backbone(right)
        f = torch.cat([f_left, f_right], dim=1)
        out_total = self.head_total(f)
        out_gdm   = self.head_gdm(f)
        out_green = self.head_green(f)
        out = torch.cat([out_total, out_gdm, out_green], dim=1)
        return out, (out_total, out_gdm, out_green)

class SimpleNet3Head(nn.Module):
    def __init__(self):
        super().__init__()
        # Shared feature extractor
        self.features = nn.Sequential(
            nn.Conv2d(3, 16, 3, stride=2, padding=1),
            nn.BatchNorm2d(16), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(16, 32, 3, stride=2, padding=1),
            nn.BatchNorm2d(32), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, stride=2, padding=1),
            nn.BatchNorm2d(64), nn.ReLU(),
            nn.AdaptiveAvgPool2d((1, 1))
        )

        self.flatten = nn.Flatten()
        self.fc_shared = nn.Linear(64, 128)

        # Regression heads (each outputs 1 value)
        self.head_total = nn.Linear(128, 1)
        self.head_gdm   = nn.Linear(128, 1)
        self.head_green = nn.Linear(128, 1)

    def forward(self, x):
        x = self.flatten(self.features(x))
        x = F.relu(self.fc_shared(x))
        out_total = self.head_total(x)
        out_gdm   = self.head_gdm(x)
        out_green = self.head_green(x)
        out = torch.cat([out_total, out_gdm, out_green], dim=1)
        return out, (out_total, out_gdm, out_green)




In [ ]:
criterion = WeightedBiomassLoss(loss_weights)
scorer = CompetitionScorer(r2_weights)
model = TwoStreamConvNeXt3Head(MODEL_NAME, PRETRAINED)
model = train_one_stage(model, train_loader, val_loader, criterion, scorer)
validate(model, val_loader, criterion, scorer, n_show=5)


Epoch 1:  18%|█▊        | 16/90 [00:26<01:49,  1.49s/batch]

In [41]:
criterion = WeightedBiomassLoss(loss_weights)
scorer = CompetitionScorer(r2_weights)
model = SimpleNet3Head()
model = train_one_stage(model, train_loader, val_loader, criterion, scorer)
validate(model, val_loader, criterion, scorer, n_show=5)

Evaluating validation set: 100%|██████████| 90/90 [00:27<00:00,  3.28batch/s]


[Epoch 1] Train=31.1585 | Val=30.6959 | MSE=1238.8955 | R²=-1.7933


Evaluating validation set:  72%|███████▏  | 65/90 [00:18<00:07,  3.44batch/s]


KeyboardInterrupt: 